In [5]:
import pandas as pd

In [1]:
import sqlite3
# sqlite3 ships with Python's standard library — no pip install needed

In [2]:
from sqlalchemy import create_engine

engine = create_engine('sqlite:///data/db_delays.db')

In [7]:
# --- 2. Load cleaned CSVs ---
dim_station = pd.read_csv('data/processed/dim_station.csv')
dim_line = pd.read_csv('data/processed/dim_line.csv')
dim_date = pd.read_csv('data/processed/dim_date.csv', parse_dates=['date'])
fact_delays = pd.read_csv('data/processed/fact_delays.csv', parse_dates=[
    'arrival_plan', 'departure_plan', 'arrival_change', 'departure_change'
])
print("All tables loaded into SQLite")

All tables loaded into SQLite


In [11]:
dim_station.to_sql('dim_station', engine, if_exists='replace', index=False)
dim_line.to_sql('dim_line', engine, if_exists='replace', index=False)
dim_date.to_sql('dim_date', engine, if_exists='replace', index=False)

9

In [12]:
# make sure column names are correct
dim_station = dim_station.rename(columns={'lat': 'latitude', 'long': 'longitude'})


In [13]:
# reapply the deduplication step from before
fact_delays_sorted = fact_delays.sort_values(
    by=['stop_id', 'arrival_change', 'departure_change'],
    na_position='first'
)
fact_delays_dedup = fact_delays_sorted.drop_duplicates(subset='stop_id', keep='last')

print(f"fact_delays: {len(fact_delays)} → fact_delays_dedup: {len(fact_delays_dedup)}")

fact_delays: 2061357 → fact_delays_dedup: 2029894


In [14]:
engine = create_engine('sqlite:///data/db_delays.db')

dim_station.to_sql('dim_station', engine, if_exists='replace', index=False)
dim_line.to_sql('dim_line', engine, if_exists='replace', index=False)
dim_date.to_sql('dim_date', engine, if_exists='replace', index=False)
fact_delays_dedup.to_sql('fact_delays', engine, if_exists='replace', index=False, chunksize=5000)

print("All tables loaded into SQLite")

All tables loaded into SQLite


In [15]:
# save fact table
fact_delays_dedup.to_csv('data/processed/fact_delays_dedup.csv', index=False)

In [16]:
print(pd.read_sql('SELECT COUNT(*) FROM dim_station', engine))

   COUNT(*)
0      1996


In [17]:
print(pd.read_sql('SELECT COUNT(*) FROM dim_line', engine))

   COUNT(*)
0       296


In [18]:
print(pd.read_sql('SELECT COUNT(*) FROM dim_date', engine))

   COUNT(*)
0         9


In [19]:
#worst station
query = """
SELECT s.station, s.state, AVG(f.arrival_delay_m) AS avg_delay, COUNT(*) AS n_stops
FROM fact_delays f
JOIN dim_station s ON f.station_id = s.station_id
GROUP BY s.station, s.state
HAVING COUNT(*) >= 20
ORDER BY avg_delay DESC
LIMIT 15
"""
pd.read_sql(query, engine)

,station,state,avg_delay,n_stops
0,Steinau (Straße),Hessen,6.169811,265
1,Flieden,Hessen,5.785408,233
2,Langenselbold,Hessen,5.287846,469
3,St. Goar,Rheinland-Pfalz,5.120370,324
4,Neuhof (Kr Fulda),Hessen,4.993243,296
5,Bad Soden-Salmünster,Hessen,4.975232,323
6,Schlüchtern,Hessen,4.791005,378
7,Rodenbach bei Hanau,Hessen,4.688525,244
8,Sonthofen,Bayern,4.687773,458
9,Teisendorf,Bayern,4.666667,297


In [24]:
import pandas as pd
import os

os.makedirs('sql/analysis', exist_ok=True)

In [25]:
#worst line
query_02 = """
SELECT l.line, l.train_type,
       AVG(f.arrival_delay_m) AS avg_delay,
       COUNT(*) AS n_stops
FROM fact_delays f
JOIN dim_line l ON f.line_id = l.line_id
GROUP BY l.line, l.train_type
HAVING COUNT(*) >= 20
ORDER BY avg_delay DESC
LIMIT 15
"""

with open('sql/analysis/02_worst_lines.sql', 'w') as f:
    f.write(query_02)

worst_lines = pd.read_sql(query_02, engine)
worst_lines

,line,train_type,avg_delay,n_stops
0,RE25,RE,5.153263,1318
1,RE23,RE,4.099237,131
2,RE85,RE,3.716617,674
3,RE5,RE,3.630995,3065
4,27,None,3.415352,9445
5,RE7,RE,3.225000,40
6,10b,None,3.211155,1004
7,RB93,RB,2.984987,1199
8,50,None,2.912529,10312
9,RB17,RB,2.846780,1351


In [26]:
# Query 3 — Delay by hour and weekday
query_03 = """
SELECT d.weekday_name, f.hour,
       AVG(f.arrival_delay_m) AS avg_delay,
       COUNT(*) AS n_stops
FROM fact_delays f
JOIN dim_date d ON f.date_id = d.date_id
GROUP BY d.weekday_name, f.hour
ORDER BY d.weekday_name, f.hour
"""

with open('sql/analysis/03_delay_by_hour_weekday.sql', 'w') as f:
    f.write(query_03)

hour_weekday = pd.read_sql(query_03, engine)
hour_weekday

,weekday_name,hour,avg_delay,n_stops
0,None,NaN,0.000000,211355
1,Friday,0.0,1.787500,6000
2,Friday,1.0,2.213093,2215
3,Friday,2.0,1.196833,442
4,Friday,3.0,0.314917,543
...,...,...,...,...
164,Wednesday,19.0,1.936127,13887
165,Wednesday,20.0,1.805812,12766
166,Wednesday,21.0,1.596974,11436
167,Wednesday,22.0,1.507679,10288


In [27]:
#Query 4 — Delay by state
query_04 = """
SELECT s.state,
       AVG(f.arrival_delay_m) AS avg_delay,
       COUNT(*) AS n_stops
FROM fact_delays f
JOIN dim_station s ON f.station_id = s.station_id
GROUP BY s.state
ORDER BY avg_delay DESC
"""

with open('sql/analysis/04_delay_by_state.sql', 'w') as f:
    f.write(query_04)

by_state = pd.read_sql(query_04, engine)
by_state

,state,avg_delay,n_stops
0,Rheinland-Pfalz,1.857736,77602
1,Bayern,1.830112,324149
2,Nordrhein-Westfalen,1.595756,337076
3,Baden-Württemberg,1.449648,249382
4,Hessen,1.252411,197737
5,Sachsen-Anhalt,1.102287,30258
6,Niedersachsen,1.062320,81114
7,Brandenburg,0.973587,57698
8,Saarland,0.906948,17055
9,Sachsen,0.881951,83364


In [28]:
#Overall punctuality baseline
query_05 = """
SELECT 
    ROUND(AVG(CASE WHEN arrival_delay_m <= 6 THEN 1.0 ELSE 0 END) * 100, 2) AS pct_on_time,
    ROUND(AVG(arrival_delay_m), 2) AS avg_delay_minutes,
    COUNT(*) AS total_stops
FROM fact_delays
"""

with open('sql/analysis/05_punctuality_overall.sql', 'w') as f:
    f.write(query_05)

overall = pd.read_sql(query_05, engine)
overall

,pct_on_time,avg_delay_minutes,total_stops
0,95.65,1.19,2029894
